In [3]:
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_platform_name", "cpu")
from jax import vmap, jit
import matplotlib.pyplot as plt
import joblib
from qdots_qll.models import game
from qdots_qll import all_funcs
import seaborn as sns
import pandas as pd
import scipy
from functools import reduce
import os
import re
from scipy.stats import binned_statistic

import seaborn as sns


import matplotlib.font_manager as font_manager

import equinox as eqx

# from matplotlib import rcParams
from scipy.stats import binned_statistic


font = {"family": "Inter"}  # , 'weight': 'normal', 'size': 12}

# Aplica la fuente definida a Matplotlib
plt.rc("font", **font)

sns.set_palette("colorblind")

In [4]:
names_true = [
    "$\\gamma ( + \\eta)$",
    "$S ( - \\eta)$",
    "$S ( +\\eta)$",
]
names_hat = [
    "$\\hat{\\gamma} ( + \\eta)$",
    "$\\hat{S} ( - \\eta)$",
    "$\\hat{S} ( +\\eta)$",
]

In [5]:
def compute_mean_one_run(cum_times_i, cov_arr_i, limites_bins):
    indices_bins = np.digitize(cum_times_i, limites_bins)
    means = np.array(
        [
            np.nanmean(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )

    std_devs = np.array(
        [
            np.nanstd(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )
    return means, std_devs


def get_binned_results_from_runs(
    times_array,
    cov_array,
    est_array,
    bin_step,
):
    dims_cov = cov_array[:, 1:].shape[2:]
    dims_est = est_array[:, 1:].shape[2:]

    cum_times = np.array(times_array[:, 1:]).cumsum(axis=1)
    cum_times_flatten = cum_times.flatten()
    cov_flatten = np.array(cov_array[:, 1:]).reshape(-1, *dims_cov)
    est_flatten = np.array(est_array[:, 1:]).reshape(-1, *dims_est)

    bins = np.arange(
        cum_times_flatten.min(), cum_times_flatten.max() + 1, bin_step
    )

    cov_mean_list = []
    cov_std_list = []

    for i in range(dims_cov[0]):
        cov_mean_row_list = []
        cov_std_row_list = []

        for j in range(dims_cov[1]):

            cov_mean_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="mean",
                    bins=bins,
                )[0]
            )
            cov_std_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="std",
                    bins=bins,
                )[0]
            )
        cov_mean_list.append(cov_mean_row_list)
        cov_std_list.append(cov_std_row_list)
    cov_mean_list = np.array(cov_mean_list).transpose(2, 1, 0)
    cov_std_list = np.array(cov_std_list).transpose(2, 1, 0)

    est_mean_list = []
    est_std_list = []

    for i in range(dims_est[0]):

        est_mean_list.append(
            binned_statistic(
                cum_times_flatten,
                est_flatten[
                    :,
                    i,
                ],
                statistic="mean",
                bins=bins,
            )[0]
        )
        est_std_list.append(
            binned_statistic(
                cum_times_flatten,
                est_flatten[
                    :,
                    i,
                ],
                statistic="std",
                bins=bins,
            )[0]
        )
    est_mean_list = np.array(est_mean_list).transpose(1, 0)
    est_std_list = np.array(est_std_list).transpose(1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times_flatten, cum_times_flatten, statistic="mean", bins=bins
    )
    return (
        cum_times_binned,
        cov_mean_list,
        cov_std_list,
        est_mean_list,
        est_std_list,
    )

In [6]:
def results_bin_single_run(times, covs, ests, no_bins):

    cum_times = np.array(times[1:]).cumsum()
    # bins = np.linspace(cum_times.min(), cum_times.max() + 1, no_bins)
    dims_cov = covs.shape[-2:]
    dims_est = ests.shape[-1:]

    cov_mean_list = []
    cov_std_list = []

    for i in range(dims_cov[0]):
        cov_mean_row_list = []
        cov_std_row_list = []

        for j in range(dims_cov[1]):

            cov_mean_row_list.append(
                binned_statistic(
                    cum_times,
                    covs[1:, i, j],
                    statistic="mean",
                    bins=no_bins,
                )[0]
            )
            cov_std_row_list.append(
                binned_statistic(
                    cum_times,
                    covs[1:, i, j],
                    statistic="std",
                    bins=no_bins,
                )[0]
            )
        cov_mean_list.append(cov_mean_row_list)
        cov_std_list.append(cov_std_row_list)
    cov_mean_list = np.array(cov_mean_list).transpose(2, 1, 0)
    cov_std_list = np.array(cov_std_list).transpose(2, 1, 0)

    est_mean_list = []
    est_std_list = []

    for i in range(dims_est[0]):

        est_mean_list.append(
            binned_statistic(
                cum_times,
                ests[
                    1:,
                    i,
                ],
                statistic="mean",
                bins=no_bins,
            )[0]
        )
        est_std_list.append(
            binned_statistic(
                cum_times,
                ests[
                    1:,
                    i,
                ],
                statistic="std",
                bins=no_bins,
            )[0]
        )
    est_mean_list = np.array(est_mean_list).transpose(1, 0)
    est_std_list = np.array(est_std_list).transpose(1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times, cum_times, statistic="mean", bins=no_bins
    )
    times_binned, _, _ = binned_statistic(
        cum_times, np.array(times[1:]), statistic="mean", bins=no_bins
    )
    return (
        times_binned,
        cum_times_binned,
        cov_mean_list,
        cov_std_list,
        est_mean_list,
        est_std_list,
    )

In [7]:
# i = 4
# run = runs[i]

# k = 0

# times_list = []
# cumtimes_list = []
# covmean_list = []
# covstd_list = []
# estmean_list = []
# eststd_list = []

# no_runs = run.iteration.shape[0]
# for k in range(no_runs):
#     covs = run.cov_array[k]
#     ests = run.estimates_array[k]
#     times = run.times_array[k]

#     re = results_bin_single_run(times, covs, ests, 2300)

#     times_list.append(re[0])
#     cumtimes_list.append(re[1])
#     covmean_list.append(re[2])
#     covstd_list.append(re[3])
#     estmean_list.append(re[4])
#     eststd_list.append(re[5])

# times_list = np.array(times_list).mean(axis=0)
# cumtimes_list = np.array(cumtimes_list).mean(axis=0)
# covmean_list = np.array(covmean_list).mean(axis=0)
# covstd_list = np.array(covstd_list).mean(axis=0)
# estmean_list = np.array(estmean_list).mean(axis=0)
# eststd_list = np.array(eststd_list).mean(axis=0)

In [8]:
# plt.plot(cumtimes_list, covmean_list[:, 0, 0])
# plt.fill_between(
#     cumtimes_list,
#     covstd_list[:, 0, 0] + covmean_list[:, 0, 0],
#     -covstd_list[:, 0, 0] + covmean_list[:, 0, 0],
#     alpha=0.4,
# )

# plt.loglog()

In [9]:
# _, _, _, a, b = get_binned_results_from_runs(
#     run.times_array, run.cov_array, run.estimates_array, 200
# )

# for

In [10]:
filenames = sorted(os.listdir("../results_cluster/one_qdot/"))
filenames = ["../results_cluster/one_qdot/" + i for i in filenames]
job_filenames = list(filter(re.compile(".*job").match, filenames))
log_filenames = list(filter(re.compile(".*log").match, filenames))

runs = [joblib.load(i) for i in job_filenames]

In [11]:
log_filenames

In [12]:
legend_array = [
    "Random",
    "Max det(FIM)",
    "Max trace(FIM)",
    "Random",
    "Max det(FIM)",
    "Max trace(FIM)",
]

In [13]:
# Now we are gonna compute the fisher information with the expected particle for each case

In [14]:
from qdots_qll.models.models_scratch_for_drafting import (
    SingleQDot3Params,
)
from qdots_qll.utils.povms import sigmas_povm
import qutip as qt

ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()
model = SingleQDot3Params(POVM_array=jnp.array(sigmas_povm))

m = model

true_pars = jnp.array([0.35833, 0.053851, -0.333695])

In [15]:
from qdots_qll.distributions import (
    initialize_particle_locations_normal_prior,
    est_cov,
)


bnds = np.array([[0.01, 0.9], [0.001, 0.22], [-0.01, -0.9]])
sigmasprior = 5
covs_prior = jnp.diagflat(jnp.std(bnds, axis=1) / sigmasprior) ** 2
mus_prior = jnp.mean(bnds, axis=1)

print(f"mus prior: {mus_prior}")
print(f"covs prior: {np.diag(covs_prior)}")

I = np.diag(np.diag(1 / covs_prior))
print(I)


seed = 10
key = jax.random.PRNGKey(seed)
no_particles = 500

init_locations = initialize_particle_locations_normal_prior(
    key, no_of_particles=no_particles, boundaries=bnds, sigmas=5
)

init_weights = jnp.ones(no_particles) / no_particles

I = np.linalg.inv(est_cov(init_locations, init_weights))
print(I)

In [16]:
est_cov(init_locations, init_weights)

In [17]:
runs[0]

In [209]:
# # i = 4

# results_binned = []

# for i in range(len(runs)):
#     (
#         cum_times_binned,
#         cov_mean_list,
#         cov_std_list,
#         est_mean_list,
#         est_std_list,
#     ) = get_binned_results_from_runs(
#         runs[i].times_array, runs[i].cov_array, runs[i].estimates_array, 1000
#     )

#     cov_dict = {
#         "times": cum_times_binned,
#         "covmean": cov_mean_list,
#         "covstd": cov_std_list,
#         "estmean": est_mean_list,
#         "eststd": est_std_list,
#     }
#     results_binned.append(cov_dict)


# for i in range(len(results_binned)):
#     cumtimes = results_binned[i]["times"]
#     times = np.diff(cumtimes, prepend=0)
#     covmeans = results_binned[i]["covmean"]
#     estmeans = results_binned[i]["estmean"]

#     fisher_information_each_time = np.array(
#         jax.vmap(
#             lambda particle, time: m.fim(particle, time, ground_state_qdot),
#             in_axes=(0, 0),
#         )(estmeans, times)
#     )

#     det_fisher_information_each_time = np.array(
#         jax.vmap(lambda f: jnp.linalg.det(f))(fisher_information_each_time)
#     )

#     results_binned[i]["fishermeans"] = fisher_information_each_time
#     results_binned[i]["fisherdets"] = det_fisher_information_each_time

#     det_covs = np.array(jax.vmap(lambda f: jnp.linalg.det(f))(covmeans))
#     results_binned[i]["covdet"] = det_covs

In [18]:
results_binned = []

no_bins = 3000
for i in range(len(runs)):
    run = runs[i]

    times_list = []
    cumtimes_list = []
    covmean_list = []
    covstd_list = []
    estmean_list = []
    eststd_list = []

    no_runs = run.iteration.shape[0]
    for k in range(no_runs):
        covs = run.cov_array[k]
        ests = run.estimates_array[k]
        times = run.times_array[k]

        re = results_bin_single_run(times, covs, ests, no_bins)

        times_list.append(re[0])
        cumtimes_list.append(re[1])
        covmean_list.append(re[2])
        covstd_list.append(re[3])
        estmean_list.append(re[4])
        eststd_list.append(re[5])

    times_list = np.array(times_list).mean(axis=0)
    cumtimes_list = np.array(cumtimes_list).mean(axis=0)
    covstd_list = np.array(covmean_list).std(axis=0)
    covmean_list = np.array(covmean_list).mean(axis=0)

    # covstd_list = np.array(covstd_list).mean(axis=0)

    eststd_list = np.array(estmean_list).std(axis=0)
    estmean_list = np.array(estmean_list).mean(axis=0)

    # eststd_list = np.array(eststd_list).mean(axis=0)

    # (
    #     cum_times_binned,
    #     cov_mean_list,
    #     cov_std_list,
    #     est_mean_list,
    #     est_std_list,
    # ) = get_binned_results_from_runs(
    #     runs[i].times_array, runs[i].cov_array, runs[i].estimates_array, 1000
    # )

    cov_dict = {
        "times": times_list,
        "cumtimes": cumtimes_list,
        "covmean": covmean_list,
        "covstd": covstd_list,
        "estmean": estmean_list,
        "eststd": eststd_list,
    }
    results_binned.append(cov_dict)


for i in range(len(results_binned)):
    times = results_binned[i]["times"]
    # times = np.diff(cumtimes, prepend=0)
    covmeans = results_binned[i]["covmean"]
    estmeans = results_binned[i]["estmean"]

    fisher_information_each_time = np.array(
        jax.vmap(
            lambda particle, time: m.fim(particle, time, ground_state_qdot),
            in_axes=(0, 0),
        )(estmeans, times)
    )

    det_fisher_information_each_time = np.array(
        jax.vmap(lambda f: jnp.linalg.det(f))(fisher_information_each_time)
    )

    results_binned[i]["fishermeans"] = fisher_information_each_time
    results_binned[i]["fisherdets"] = det_fisher_information_each_time

    det_covs = np.array(jax.vmap(lambda f: jnp.linalg.det(f))(covmeans))
    results_binned[i]["covdet"] = det_covs

In [19]:
# Plot here the optimal determinant

best_t = runs[4].times_array.mean()
true_inv_FIM = jnp.linalg.inv(m.fim(true_pars, best_t, ground_state_qdot))
true_FIM = m.fim(true_pars, best_t, ground_state_qdot)

In [73]:
for i in range(len(results_binned)):
    run = runs[i]

    times = np.array(run.times_array).flatten()

    rng = np.random.default_rng(seed=0)

    idx = rng.choice(np.arange(len(times)), size=10000)

    ntimes = times[idx]

    nestmeans = np.array(run.estimates_array).reshape(-1, 3)[idx]

    ncovmeans = np.array(run.cov_array).reshape(-1, 3, 3)[idx]

    ncumtimes = np.array(run.times_array.cumsum(axis=1).flatten())[idx]

    ord_idx = np.argsort(ncumtimes)

    ncumtimes = ncumtimes[ord_idx]
    ntimes = ntimes[ord_idx]
    nestmeans = nestmeans[ord_idx]

    fisher_information_each_time = np.array(
        jax.vmap(
            lambda particle, time: model.fim(
                particle, time, ground_state_qdot
            ),
            in_axes=(0, 0),
        )(nestmeans, ntimes)
    )

    det_fisher_information_each_time = np.array(
        jax.vmap(lambda f: jnp.linalg.det(f))(fisher_information_each_time)
    )

    results_binned[i]["cgfishermeans"] = fisher_information_each_time
    results_binned[i]["cgfisherdets"] = det_fisher_information_each_time

    cgdet_covs = np.array(jax.vmap(lambda f: jnp.linalg.det(f))(ncovmeans))

    results_binned[i]["cgcovmeans"] = ncovmeans
    results_binned[i]["cgcovdet"] = cgdet_covs

    results_binned[i]["cgcumtimes"] = ncumtimes

for i in range(len(results_binned)):
    times = results_binned[i]["times"]
    # times = np.diff(cumtimes, prepend=0)
    covmeans = results_binned[i]["covmean"]
    estmeans = results_binned[i]["estmean"]

    fisher_information_each_time = np.array(
        jax.vmap(
            lambda particle, time: m.fim(particle, time, ground_state_qdot),
            in_axes=(0, 0),
        )(estmeans, times)
    )

    det_fisher_information_each_time = np.array(
        jax.vmap(lambda f: jnp.linalg.det(f))(fisher_information_each_time)
    )

    results_binned[i]["fishermeans"] = fisher_information_each_time
    results_binned[i]["fisherdets"] = det_fisher_information_each_time

    det_covs = np.array(jax.vmap(lambda f: jnp.linalg.det(f))(covmeans))
    results_binned[i]["covdet"] = det_covs

In [74]:
fig, ax = plt.subplots(dpi=600)

for i in range(3, 6, 1):
    # print(legend_array[i])
    ax.plot(
        results_binned[i]["cgcumtimes"],
        results_binned[i]["cgfisherdets"],
        ".",
        label=legend_array[i],
        ms=2,
    )
ax.axhline(
    jnp.linalg.det(true_FIM),
    label="$\\operatorname{det} (F( \\theta_{True}, t_{det max})_{ij} ) $",
)
ax.legend()
ax.set_ylim(1e-5, 0.3)

ax.loglog()

ax.set_xlabel("Total experimental time (ps)")
ax.set_ylabel("$|F( E[\\theta_k], t)_{ij} | $")
plt.suptitle(
    " $\\operatorname{det} (F( E[\\theta_k], t)_{ij} ). $ \nSetup: Single dot. POVM sigma basis. "
)
plt.tight_layout()
plt.show()

In [75]:
results_binned[0].keys()

In [69]:
jnp.linalg.inv(results_binned[0]['covmean'][30])

In [83]:
fig, axs = plt.subplots(1, 3, figsize=(10, 5), dpi=600)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]

# for j, ax in enumerate(axs.flatten()):
for j in range(3):
    ax = axs.flatten()[j]
    for i in range(3, 6, 1):
        # i=4
        ax.set_title(names_true[j])
        ax.set_xlabel("Total experimental time (ps)")
        ax.set_ylim(1e-6, 1e-2)
        ax.plot(
            results_binned[i]["cumtimes"],
            results_binned[i]["covmean"][:, j, j],
            "-",
            color=colors[i - 3],
            label=legend_array[i],
        )

        ax.fill_between(
            results_binned[i]["cumtimes"],
            results_binned[i]["covmean"][:, j, j]
            + results_binned[i]["covstd"][:, j, j],
            results_binned[i]["covmean"][:, j, j]
            - results_binned[i]["covstd"][:, j, j],
            alpha=0.2,
        )

        # ax.plot(
        #     results_binned[i]["cumtimes"],
        #     jax.vmap(
        #         (
        #             lambda T, EF, I: 1
        #             / T
        #             * 1
        #             * jnp.linalg.inv(1 * (1 * EF + 1 * I / T))
        #         ),
        #         in_axes=(0, 0, None),
        #     )(
        #         results_binned[i]["cumtimes"],
        #         results_binned[i]["fishermeans"],
        #         I,
        #     )[
        #         :, j, j
        #     ],
        #     "-.",
        #     ms=3,
        #     alpha=0.7,
        #     color=colors[i - 3],
        # )
        ax.plot(
            results_binned[i]["cumtimes"],
            jax.vmap(
                (
                    lambda T, EF, I: 1
                    / T
                    * 1
                    * jnp.linalg.inv(1 * (1 * EF + 1 * jnp.linalg.inv(I) / T))
                ),
                in_axes=(0, 0, 0),
            )(
                results_binned[i]["cumtimes"],
                results_binned[i]["fishermeans"],
                results_binned[i]["covmean"],
            )[
                :, j, j
            ],
            "-.",
            ms=3,
            alpha=0.7,
            color=colors[i - 3],
        )

        ax.loglog()
    # ax.plot(
    #     results_binned[i]["cumtimes"],
    #     jax.vmap(
    #         (
    #             lambda T, EF, I: 1
    #             / T
    #             * 1
    #             * jnp.linalg.inv(1 * (1 * EF + 1 * I / T))
    #         ),
    #         in_axes=(0, None, None),
    #     )(results_binned[i]["cumtimes"], true_FIM, I,)[:, j, j],
    #     "-",
    #     ms=3,
    #     alpha=0.7,
    #     color="black",
    #     label="Best scaling",
    # )

    ax.legend()

fig.suptitle(
    "Evolution of $Cov(\\theta_i)$ \nDotted line corresponds to the VT bound \ncomputed with $F( E[\\theta_k], t_k)_{ij}$ at experimental time k.\n Averaged with 20 runs\nSetup: Single dot. POVM sigma basis."
)
plt.tight_layout()
plt.show()

In [1]:
from sklearn.metrics import mean_squared_error

In [2]:
for i in range(3, 6, 1):

    print(
        np.sqrt(
            mean_squared_error(results_binned[i]["fishermeans"][-1], true_FIM)
        )
    )
    # print(results_binned[i]["fishermeans"][-1])

In [95]:
results_binned[i].keys()

In [43]:
fig, ax = plt.subplots(dpi=200)


colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]

for i in range(3, 6, 1):
    ax.plot(
        results_binned[i]["cumtimes"],
        results_binned[i]["covdet"][:],
        "-.",
        label=legend_array[i],
    )

    ax.plot(
        results_binned[i]["cumtimes"],
        jax.vmap(
            (
                lambda T, EF, I: 1
                / T**3
                / jnp.linalg.det((1 * (1 * EF + 1 * I / T)))
            ),
            in_axes=(0, 0, None),
        )(
            results_binned[i]["cumtimes"],
            results_binned[i]["fishermeans"],
            I,
        )[
            :,
        ],
        "-",
        ms=3,
        alpha=0.7,
        color=colors[i - 3],
    )
    ax.legend()
    ax.loglog()

    # ax.set_xlim(100, 10000)

In [48]:
fig, axs = plt.subplots(1, 3, figsize=(10, 5), dpi=400)


for j, ax in enumerate(axs.flatten()):
    run = runs[j]
    ax.hist(np.array(run.times_array).flatten(), bins=100)
    ax.set_title(legend_array[j])
    ax.set_xlabel("Time (ps)")


fig.suptitle(f"Histogram of selected times. 20 runs. \nSetup: Single dot. POVM sigma basis.")
plt.tight_layout()
plt.show()